# CircuitGraft - D1 / G0 Smoke Test on Tiny Aya (Colab GPU)

This notebook runs the official **D1 / G0 smoke test** per `SPEC.md` §12:
- Runs `safetune`, `circuitkit`, `auditkit`, and `curatorkit` on real `CohereLabs/tiny-aya-global`.
- Uses CUDA acceleration (e.g. T4 / A100 / V100 GPU).
- Generates official G0 evidence under `results/d1/smoke/` (`summary.json`, `summary.md`, and toolkit results).

> **Prerequisites:** Accepted licence for `CohereLabs/tiny-aya-global` and `CohereLabs/tiny-aya-base` on Hugging Face.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone Repository or Set Up Workspace
If you are uploading the repository zip or cloning directly from GitHub, navigate to the repo root.

In [ ]:
import os

# If running in Colab from cloned repo:
if not os.path.exists('scripts/d1_smoke.py'):
    # Clone your repo or upload files
    print('Please ensure you are in the circuit-graft root directory.')
else:
    print('Repository root verified:', os.getcwd())

## 3. Install Required Toolkits

In [ ]:
!pip install --upgrade pip
!pip install safetune==0.1.12 circuitkit==0.2.0 auditkit==1.1.3 "curatorkit[generation,hf]==0.1.22"
!pip install torch transformers transformer_lens peft trl sentence_transformers litellm python-dotenv typer rich

## 4. Set Hugging Face Token
Use Colab Secrets (key: `HF_TOKEN`) or enter your token interactively.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if not token:
    token = getpass('Enter your Hugging Face token (with Tiny Aya access): ')

os.environ['HF_TOKEN'] = token.strip()
os.environ['HF_ORG'] = 'dubeydevansh'
print('HF_TOKEN configured.')

## 5. Run D1 Smoke Test on CUDA (Real Tiny Aya)
This runs preflight and all 4 toolkits on `CohereLabs/tiny-aya-global`.

In [ ]:
!python scripts/d1_smoke.py --device cuda --output-dir results/d1/smoke

## 6. Inspect Summary and G0 Gate Status

In [ ]:
from IPython.display import Markdown, display
with open('results/d1/smoke/summary.md', 'r') as f:
    display(Markdown(f.read()))

## 7. Package G0 Evidence for Download
Download this archive to commit into `results/d1/smoke/` in the local repo.

In [ ]:
!zip -r d1_smoke_results.zip results/d1/smoke
try:
    from google.colab import files
    files.download('d1_smoke_results.zip')
except Exception as e:
    print('Archive created at d1_smoke_results.zip:', e)